# Day 56 · Exercise 2: File Validation

**What you'll build:** Implement `validate_upload(content, filename, allowed_extensions, max_bytes)` returning `(True, '')` for valid files and `(False, reason)` for invalid ones. Validation runs BEFORE storage — never save what you haven't checked.

## Setup (provided)

In [ ]:
from pathlib import Path


## Your Implementation

In [ ]:
def validate_upload(content: bytes, filename: str,
                    allowed_extensions: list[str], max_bytes: int) -> tuple[bool, str]:
    """Validate an uploaded file's size and extension.

    Args:
        content:             Raw file bytes.
        filename:            Original filename (e.g. 'report.pdf').
        allowed_extensions:  Whitelist of lowercase extensions (e.g. ['.txt', '.pdf']).
        max_bytes:           Maximum allowed file size in bytes.
    Returns:
        (True, "") if valid.
        (False, reason_str) if invalid.
    """
    # TODO:
    # 1. Return (False, "File is empty") if len(content) == 0
    # 2. Return (False, "File too large ...") if len(content) > max_bytes
    # 3. Extract extension: Path(filename).suffix.lower()
    # 4. Return (False, "Extension ... not allowed ...") if ext not in allowed_extensions
    # 5. Return (True, "")
    raise NotImplementedError


In [ ]:
def validate_upload(content: bytes, filename: str,
                    allowed_extensions: list[str], max_bytes: int) -> tuple[bool, str]:
    if len(content) == 0:
        return False, "File is empty"
    if len(content) > max_bytes:
        return False, f"File too large ({len(content)} bytes, max {max_bytes})"
    ext = Path(filename).suffix.lower()
    if ext not in allowed_extensions:
        return False, f"Extension '{ext}' not allowed (allowed: {allowed_extensions})"
    return True, ""


## Check Your Work

In [ ]:
def _run_checks():
    score = 0
    total = 5

    def _chk(n, ok, msg):
        nonlocal score
        print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
        if ok:
            score += 1

    allowed = [".txt", ".pdf", ".md"]
    max_b   = 1000

    try:
        ok, msg = validate_upload(b"hello", "doc.txt", allowed, max_b)
    except NotImplementedError:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: validate_upload not implemented")
        print(f"\nScore: 0 / {total}")
        return

    _chk(1, ok is True and msg == "",
         f"valid .txt → (True, '') (got ({ok}, '{msg}'))")

    ok2, msg2 = validate_upload(b"", "doc.txt", allowed, max_b)
    _chk(2, ok2 is False and "empty" in msg2.lower(),
         f"empty file → (False, ...'empty'...) (got ({ok2}, '{msg2}'))")

    big = b"x" * (max_b + 1)
    ok3, msg3 = validate_upload(big, "doc.txt", allowed, max_b)
    _chk(3, ok3 is False and "large" in msg3.lower(),
         f"too-large file → (False, ...'large'...) (got ({ok3}, '{msg3}'))")

    ok4, msg4 = validate_upload(b"data", "image.png", allowed, max_b)
    _chk(4, ok4 is False and "png" in msg4.lower(),
         f".png rejected → (False, ...) (got ({ok4}, '{msg4}'))")

    ok5, msg5 = validate_upload(b"data", "report.pdf", allowed, max_b)
    _chk(5, ok5 is True,
         f"valid .pdf → (True, '') (got ({ok5}, '{msg5}'))")

    print(f"\nScore: {score} / {total}")
    if score == total:
        print("🎉 Exercise complete!")

_run_checks()


## Bonus Challenge

Extend `validate_upload` with a content-sniffing check: for `.pdf` files, verify the first 5 bytes equal `b'%PDF-'`. This catches files renamed to `.pdf` that are actually images or executables. A well-formed PDF always starts with the magic bytes `%PDF-`.

## Solution

<details>
<summary>Show solution</summary>

```python
def validate_upload(content: bytes, filename: str,
                    allowed_extensions: list[str], max_bytes: int) -> tuple[bool, str]:
    if len(content) == 0:
        return False, "File is empty"
    if len(content) > max_bytes:
        return False, f"File too large ({len(content)} bytes, max {max_bytes})"
    ext = Path(filename).suffix.lower()
    if ext not in allowed_extensions:
        return False, f"Extension '{ext}' not allowed (allowed: {allowed_extensions})"
    return True, ""
```

**Why this works:** The function checks in order: empty → too large → wrong
extension. `Path(filename).suffix.lower()` extracts the extension reliably —
`Path("report.PDF").suffix.lower()` → `".pdf"`. The whitelist approach is safer
than a blacklist: you explicitly allow what you want and reject everything else.
Return a tuple rather than raising, so the caller can decide how to surface the
error (HTTP 400, log, etc.).

</details>